# 03. Continuous Control Ddpg Td3 and SAC

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Lillicrap et al. (2015 DDPG); Fujimoto et al. (2018 TD3); Haarnoja et al. (2018 SAC).
- **Purpose:** First-principles mathematical implementation and empirical verification of modern continuous actor-critic control algorithms: Deep Deterministic Policy Gradient (DDPG), Twin Delayed DDPG (TD3), and Soft Actor-Critic (SAC).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. Policy Gradient Reinforce A2c PPO and GFlowNets](02_policy_gradient_reinforce_a2c_ppo_and_gflownets.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. Offline RL Imitation Cql IQL Gail and Decision Transformer](04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never omit the $\tanh$ change-of-variables Jacobian correction $\sum_i \log(1 - a_i^2 + \epsilon)$ when computing squashed Gaussian action probabilities in SAC. Omitting it biases the policy entropy gradient toward action boundaries $\pm 1.0$, causing extreme control saturation. Do not use DDPG on tasks with sharp value landscapes or high sensor noise; use TD3 with clipped double-Q and target action smoothing to prevent value divergence.

## Mathematical Foundations & Governing Equations

### 1. TD3 Target Policy Smoothing & Clipped Double-Q
$$
\tilde{a}' = \text{clip}\left( \mu_{\bar{\theta}}(s') + \text{clip}(\epsilon, -c, c), \; a_{\min}, \; a_{\max} \right), \quad \epsilon \sim \mathcal{N}(0, \sigma^2)
$$
$$
y^{\text{TD3}} = r + \gamma (1 - d) \min\left( Q_{\bar{\phi}_1}(s', \tilde{a}'), \; Q_{\bar{\phi}_2}(s', \tilde{a}') \right)
$$

### 2. SAC Squashed Gaussian Policy & Jacobian
Given $u \sim \mathcal{N}(\mu(s), \text{diag}(\sigma(s)^2))$ and $a = \tanh(u)$:
$$
\log \pi(a \mid s) = \log \mathcal{N}(u; \mu(s), \sigma(s)^2) - \sum_{i=1}^D \log\left( 1 - \tanh^2(u_i) + \epsilon \right)
$$

### 3. Temperature Parameter Optimization
$$
\mathcal{L}(\alpha) = -\hat{\mathbb{E}}_{a \sim \pi} \left[ \alpha \left( \log \pi(a \mid s) + \bar{\mathcal{H}} \right) \right]
$$

### Asymptotic Time & Memory Complexity
- TD3 Target Smoothing: $\mathcal{O}(B \cdot D)$ time and memory.
- Twin Critic Evaluation: $\mathcal{O}(B)$ elementwise comparison time.
- Squashed Gaussian Log-Probability: $\mathcal{O}(B \cdot D)$ elementwise time and memory.

In [ ]:
# Setup and seed verification environment
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def td3_target_smoothing(
    target_actions: torch.Tensor,
    noise: torch.Tensor,
    noise_clip: float = 0.5,
    action_min: float = -1.0,
    action_max: float = 1.0,
) -> torch.Tensor:
    clipped_noise = torch.clamp(noise, -noise_clip, noise_clip)
    return torch.clamp(target_actions + clipped_noise, action_min, action_max)

def td3_clipped_critic_target(
    rewards: torch.Tensor,
    dones: torch.Tensor,
    q1_target: torch.Tensor,
    q2_target: torch.Tensor,
    gamma: float = 0.99,
) -> torch.Tensor:
    min_q = torch.min(q1_target, q2_target)
    return rewards + gamma * (1.0 - dones) * min_q

def sac_squashed_gaussian_log_prob(
    u: torch.Tensor,
    mean: torch.Tensor,
    log_std: torch.Tensor,
    eps: float = 1e-6,
) -> tuple[torch.Tensor, torch.Tensor]:
    std = torch.exp(log_std)
    var = std ** 2
    log_scale = 2.0 * log_std
    gaussian_log_p = -0.5 * torch.sum(
        ((u - mean) ** 2) / var + log_scale + math.log(2.0 * math.pi),
        dim=-1,
    )
    actions = torch.tanh(u)
    jacobian = torch.sum(torch.log(torch.clamp(1.0 - actions ** 2, min=eps)), dim=-1)
    log_probs = gaussian_log_p - jacobian
    return actions, log_probs

def sac_temperature_loss(
    log_alpha: torch.Tensor,
    log_probs: torch.Tensor,
    target_entropy: float,
) -> torch.Tensor:
    alpha = torch.exp(log_alpha)
    return -torch.mean(alpha * (log_probs + target_entropy))

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-6, f"Linear anchor parity error: {err_anchor:.2e}"

smoothed = td3_target_smoothing(weights["target_actions"], weights["noise"], noise_clip=0.5, action_min=-1.0, action_max=1.0)
err_smooth = float(torch.max(torch.abs(smoothed - weights["smoothed_actions_ref"])))
assert err_smooth < 1e-6, f"TD3 target smoothing parity error: {err_smooth:.2e}"

targets = td3_clipped_critic_target(weights["rewards"], weights["dones"], weights["q1_target"], weights["q2_target"], gamma=0.99)
err_targets = float(torch.max(torch.abs(targets - weights["td3_targets_ref"])))
assert err_targets < 1e-6, f"TD3 clipped target parity error: {err_targets:.2e}"

actions, log_probs = sac_squashed_gaussian_log_prob(weights["u"], weights["mean"], weights["log_std"])
err_actions = float(torch.max(torch.abs(actions - weights["sac_actions_ref"])))
assert err_actions < 1e-6, f"SAC actions parity error: {err_actions:.2e}"
err_lp = float(torch.max(torch.abs(log_probs - weights["sac_log_probs_ref"])))
assert err_lp < 1e-6, f"SAC log probs parity error: {err_lp:.2e}"

alpha_l = sac_temperature_loss(weights["log_alpha"], weights["sac_log_probs_ref"], target_entropy=-3.0)
err_alpha = float(torch.max(torch.abs(alpha_l - weights["alpha_loss_ref"])))
assert err_alpha < 1e-6, f"SAC alpha loss parity error: {err_alpha:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_smooth, err_targets, err_actions, err_lp, err_alpha):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_smooth = benchmark_fn(
    lambda: td3_target_smoothing(weights["target_actions"], weights["noise"]),
    iters=100,
    name="TD3 Action Smoothing",
)
b_critic = benchmark_fn(
    lambda: td3_clipped_critic_target(weights["rewards"], weights["dones"], weights["q1_target"], weights["q2_target"]),
    iters=100,
    name="TD3 Clipped Double-Q Target",
)
b_sac = benchmark_fn(
    lambda: sac_squashed_gaussian_log_prob(weights["u"], weights["mean"], weights["log_std"]),
    iters=100,
    name="SAC Squashed Log-Prob",
)

print(format_benchmark_table([b_smooth, b_critic, b_sac]))

In [ ]:
# Failure Mode Contrast: Neglecting Tanh Jacobian Correction in SAC
# When an action is pushed close to the boundary (u = 2.5, a = tanh(2.5) ~ 0.9866),
# the density transformation compresses space, so log p(a) must be higher than log p(u).
u_test = torch.tensor([[2.5]])
mean_test = torch.tensor([[0.0]])
log_std_test = torch.tensor([[0.0]])

# Correct SAC log prob with Jacobian correction:
_, correct_log_p = sac_squashed_gaussian_log_prob(u_test, mean_test, log_std_test)

# Incorrect log prob omitting Jacobian:
raw_gaussian_log_p = -0.5 * (u_test ** 2 + math.log(2.0 * math.pi))

jacobian_term = float(torch.log(1.0 - torch.tanh(u_test) ** 2 + 1e-6).item())
print(f"Raw Gaussian Log Probability:     {raw_gaussian_log_p.item():.4f}")
print(f"Jacobian Correction Term:         {jacobian_term:.4f} (Severely negative near boundary!)")
print(f"Correct Corrected Log-Prob:       {correct_log_p.item():.4f}")

# Assert that omitting the Jacobian term underestimates log probability by more than 3.0 nats
diff_nats = (correct_log_p.item() - raw_gaussian_log_p.item())
assert diff_nats > 3.0, f"Omitting Jacobian must underestimate log-prob by > 3.0 nats near boundary (got {diff_nats:.2f})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Twin-Critic Minimum**: Taking $\min(Q_1, Q_2)$ systematically eliminates overestimation bias, transforming unstable continuous control into steady monotonic policy improvement.
2. **Jacobian Conservation**: Bounding actions via $\tanh$ changes the volume element; subtracting $\sum_i \log(1 - a_i^2)$ is strictly necessary to preserve probability measure conservation.
3. **Entropy Regularization**: Maximizing policy entropy prevents premature collapse into sub-optimal local deterministic policies, driving broad state-space exploration.

---
### Cross-Domain Generalisation Matrix
- **Robotics Manipulation**: SAC enables multi-fingered hand grasp policies that discover multiple distinct grasping poses for arbitrary objects.
- **Chemical Process Control**: TD3 ensures that actuator valves respond smoothly without erratic bang-bang chatter caused by critic overestimation.
- **Autonomous Vehicles**: Squashed Gaussian distributions guarantee that steering angle outputs remain strictly within physical wheel lock angles without arbitrary post-hoc clipping.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_continuous_control_ddpg_td3_and_sac.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.md)
- **PyTorch 2.x Module:** [03_continuous_control_ddpg_td3_and_sac.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.py)
- **Functional JAX Module:** [03_continuous_control_ddpg_td3_and_sac.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.py)
- **Native Rust Candle Track:** [03_continuous_control_ddpg_td3_and_sac.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.rs)
- **Hardware Kernel (CPU / WGSL):** [03_continuous_control_ddpg_td3_and_sac.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.rs)
- **Golden SafeTensors:** [03_continuous_control_ddpg_td3_and_sac.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/03_continuous_control_ddpg_td3_and_sac.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Policy Gradient Reinforce A2c PPO and GFlowNets](02_policy_gradient_reinforce_a2c_ppo_and_gflownets.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 04. Offline RL Imitation Cql IQL Gail and Decision Transformer](04_offline_rl_imitation_cql_iql_gail_and_decision_transformer.ipynb) &rarr;